In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Fact Orders

#### Reading data

In [0]:
df = spark.sql("SELECT * FROM databricks_pipeline_cata.silver.orders")
df.display()

#### Fact DataFrame

In [0]:
df_dimcust = spark.sql("SELECT dimCustomerKey, customer_id AS dim_customer_id FROM databricks_pipeline_cata.gold.dimcustomers")

df_dimprod = spark.sql("SELECT product_id AS dimProductKey, product_id AS dim_product_id FROM databricks_pipeline_cata.gold.dimproducts")

In [0]:
df_fact = df.join(df_dimcust, df["customer_id"] == df_dimcust["dim_customer_id"], "left").join(df_dimprod, df["product_id"] == df_dimprod["dim_product_id"], "left")

df_fact_new = df_fact.drop("dim_customer_id", "dim_product_id", "customer_id", "product_id")

In [0]:
df_fact_new.display()

#### Upsert on fact table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricks_pipeline_cata.gold.FactOrders"):
    dlt_org = DeltaTable.forName(spark, "databricks_pipeline_cata.gold.FactOrders")
    dlt_org.alias("trg").merge(df_fact_new.alias("src"), "trg.order_id = src.order_id AND trg.dimCustomerKey = src.dimCustomerKey AND trg.dimProductKey = src.dimProductKey")\
                .whenMatchedUpdateAll()\
                .whenNotMatchedInsertAll()\
                .execute()
else:
    df_fact_new.write.format("delta")\
                .option("path", "abfss://gold@databrickspipeline07.dfs.core.windows.net/FactOrders")\
                .saveAsTable("databricks_pipeline_cata.gold.FactOrders")

In [0]:
%sql
SELECT * FROM databricks_pipeline_cata.gold.factorders